In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import shutil
import pandas as pd

In [18]:
csv_path = r'E:\data\202502_signboard\data_annotation\docs\check0916.csv'
csv_path_save = r'E:\data\202502_signboard\data_annotation\docs\check0916_split.csv'

bd_data_dir = r'E:\data\202502_signboard\data_annotation\bd_data\data687'
bd_data_check0916_dir = r'E:\data\202502_signboard\data_annotation\bd_data\data687_check0916'

In [12]:
import pandas as pd

# 尝试常见繁体中文编码（按优先级排序）
encodings_to_try = [
    'big5',        # 台湾繁体最常用
    'cp950',       # Big5的扩展
    'gb18030',     # 兼容简繁
    'utf-8-sig',   # 带BOM的UTF-8
    'latin1'       # 最后备选
]

for enc in encodings_to_try:
    try:
        df = pd.read_csv(csv_path, encoding=enc)
        print(f"成功读取！编码: {enc}")
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("所有编码尝试失败，请检查文件")

成功读取！编码: gb18030


In [11]:
df = pd.read_csv(csv_path, encoding='gb18030')

In [13]:
def split_filename(filename):
    # 从右边分割一次 '_'，得到 "prefix_part" 和 "y.ext"
    prefix_part, y_ext = filename.rsplit('_', 1)
    
    # 再分割 "y.ext" 得到 "y" 和 "ext"
    y, ext = y_ext.split('.', 1)
    
    # 合并 "prefix_part" 和 "ext" 得到 "prefix"
    prefix = f"{prefix_part}.{ext}"
    
    return prefix, y

In [14]:


# 应用函数并拆分成两列
df[['file_name', 'object_id']] = df['object_name'].apply(lambda x: pd.Series(split_filename(x)))


In [15]:
df.to_csv(csv_path_save, index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd
import os
import shutil
from pathlib import Path
from tqdm import tqdm


def data_cp(input_dir, output_dir, img_list):
    """
    复制图片和对应的标签文件到输出目录
    
    Args:
        input_dir (str): 输入目录路径
        output_dir (str): 输出目录路径
        img_list (list): 图片文件名列表
    """
    # 构建输入输出目录的完整路径
    input_img_dir = os.path.join(input_dir, 'images')
    output_img_dir = os.path.join(output_dir, 'images')
    input_label_dir = os.path.join(input_dir, 'labels')
    output_label_dir = os.path.join(output_dir, 'labels')
    
    # 创建输出目录（如果不存在）
    os.makedirs(output_img_dir, exist_ok=True)
    os.makedirs(output_label_dir, exist_ok=True)
    
    # 使用进度条遍历所有图片文件
    for img_name in tqdm(img_list):
        # 获取对应的标签文件名（将图片扩展名改为.txt）
        label_name = Path(img_name).with_suffix('.txt')
        
        # 构建完整的输入输出路径
        input_img_path = os.path.join(input_img_dir, img_name)
        output_img_path = os.path.join(output_img_dir, img_name)
        input_label_path = os.path.join(input_label_dir, label_name)
        output_label_path = os.path.join(output_label_dir, label_name)
        
        # 复制图片文件和对应的标签文件
        shutil.copy(input_img_path, output_img_path)
        shutil.copy(input_label_path, output_label_path)

In [17]:
img_list = df['file_name'].tolist()

In [20]:
data_cp(bd_data_dir, bd_data_check0916_dir, img_list)

  0%|          | 0/93 [00:00<?, ?it/s]

100%|██████████| 93/93 [00:01<00:00, 70.01it/s]


In [27]:
import sys
sys.path.append(r'E:\repository\dataset_tools\dataformat_swift')
from yolo2xanylabeling import yolo_to_xanylabeling_dir
yolo_to_xanylabeling_dir(
    yolo_label_dir=r'E:\data\202502_signboard\data_annotation\bd_data\data687_check0916\labels', 
    images_dir=r'E:\data\202502_signboard\data_annotation\bd_data\data687_check0916\images', 
    xanylabeling_label_dir=r'E:\data\202502_signboard\data_annotation\bd_data\data687_check0916\json', 
    class_file=r'E:\data\202502_signboard\data_annotation\docs\class_c6.txt', 
    attribute_file=r'E:\data\202502_signboard\data_annotation\docs\attribute.yaml')

100%|██████████| 74/74 [00:14<00:00,  5.07it/s]


In [6]:
import os
from yolo_mask_crop import myolo_crop

root_dir = r'E:\data\202502_signboard\data_annotation\bd_data\data687_check0916'
image_dir = os.path.join(root_dir, 'images')
label_dir = os.path.join(root_dir, 'check_0916_labels_0917')
image_crop_dir = os.path.join(root_dir, 'images_crop')
class_file = os.path.join(root_dir, 'class_c6.txt')
attribute_file = os.path.join(root_dir, 'attribute.yaml')
myolo_crop(
    image_dir, label_dir, image_crop_dir, class_file, seg=True,
    attribute_file=attribute_file,
    annotation=False,
    save_method='attribute',
    only_defect=True,
    with_boundary=True,
    crop_method='with_background_image_shape'
)



100%|██████████| 862/862 [00:00<?, ?it/s]
